In [1]:
%pip install datasets

In [2]:
import numpy as np
from datasets import load_dataset


news = load_dataset("argilla/news-summary", split="test")
df = news.to_pandas().sample(5000, random_state=42)[["text", "prediction"]]
df["text"] = "summarize: " + df["text"]
df["prediction"] = df["prediction"].map(lambda x: x[0]["text"])
train, valid, test = np.split(
    df.sample(frac=1, random_state=42), [int(0.6 * len(df)), int(0.8 * len(df))]
)

print(f"Source News : {train.text.iloc[0][:200]}")
print(f"Summarization : {train.prediction.iloc[0][:50]}")
print(f"Training Data Size : {len(train)}")
print(f"Validation Data Size : {len(valid)}")
print(f"Testing Data Size : {len(test)}")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Source News : summarize: DANANG, Vietnam (Reuters) - Russian President Vladimir Putin said on Saturday he had a normal dialogue with U.S. leader Donald Trump at a summit in Vietnam, and described Trump as civil, we
Summarization : Putin says had useful interaction with Trump at Vi
Training Data Size : 3000
Validation Data Size : 1000
Testing Data Size : 1000


/usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [3]:
import torch
from transformers import T5Tokenizer
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.data import RandomSampler, SequentialSampler
from torch.nn.utils.rnn import pad_sequence


def make_dataset(data, tokenizer, device):
    source = tokenizer(
        text=data.text.tolist(),
        padding="max_length",
        max_length=128,
        pad_to_max_length=True,
        truncation=True,
        return_tensors="pt"
    )

    target = tokenizer(
        text=data.prediction.tolist(),
        padding="max_length",
        max_length=128,
        pad_to_max_length=True,
        truncation=True,
        return_tensors="pt"
    )

    source_ids = source["input_ids"].squeeze().to(device)
    source_mask = source["attention_mask"].squeeze().to(device)
    target_ids = target["input_ids"].squeeze().to(device)
    target_mask = target["attention_mask"].squeeze().to(device)
    return TensorDataset(source_ids, source_mask, target_ids, target_mask)

def get_datalodader(dataset, sampler, batch_size):
    data_sampler = sampler(dataset)
    dataloader = DataLoader(dataset, sampler=data_sampler, batch_size=batch_size)
    return dataloader


epochs = 50
batch_size = 8
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = T5Tokenizer.from_pretrained(
    pretrained_model_name_or_path="t5-small"
)

train_dataset = make_dataset(train, tokenizer, device)
train_dataloader = get_datalodader(train_dataset, RandomSampler, batch_size)

valid_dataset = make_dataset(valid, tokenizer, device)
valid_dataloader = get_datalodader(valid_dataset, SequentialSampler, batch_size)

test_dataset = make_dataset(test, tokenizer, device)
test_dataloader = get_datalodader(test_dataset, SequentialSampler, batch_size)

print(next(iter(train_dataloader)))
print(tokenizer.convert_ids_to_tokens(21603))
print(tokenizer.convert_ids_to_tokens(10))

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


[tensor([[21603,    10,   549,  ...,    13,     8,     1],
        [21603,    10,  8145,  ...,    23,   257,     1],
        [21603,    10,  6554,  ...,  7070,   166,     1],
        ...,
        [21603,    10,   301,  ...,    19,   358,     1],
        [21603,    10,     3,  ...,   282,    62,     1],
        [21603,    10,     3,  ...,   243,    34,     1]], device='cuda:0'), tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1]], device='cuda:0'), tensor([[29449,  3606,     7,  ...,     0,     0,     0],
        [ 7457,  5022, 22159,  ...,     0,     0,     0],
        [18263,   819,    13,  ...,     0,     0,     0],
        ...,
        [ 3371, 23419,   428,  ...,     0,     0,     0],
        [    3,    31, 10273,  ...,     0,     0,     0],
        [ 3371,  4297,  6914,  ...,     0,     0,     0]], device='cuda:0'), ten

In [4]:
for source_ids, source_mask, target_ids, target_mask in train_dataloader:
    print("source_ids:", source_ids[0])
    print("source_mask:", source_mask[0])
    print("target_ids:", target_ids[0])
    print("target_mask:", target_mask[0])
    break

source_ids: tensor([21603,    10,  8761, 18169,   476,    41, 18844,    61,     3,    18,
          907,  1323,  4654,  7471, 11066, 18786,    65,     3, 27692,     3,
            9,  4355,   719,    12, 30450,    30,  2089,     6,     8,   412,
            5,   134,     5,     3, 30973,    16, 30450,   243,     6,   788,
           12, 19927, 20971,     5,   105, 12998,    15,    12, 19927, 20971,
           11,     8,  1775,    13,  4654,    22,     7,  5461,    12,  8922,
           12, 24356,  1341,    12,   579,  1899,    11,   827,  3620,     6,
            8,  4355,   719,    57,  4654,  7471, 11066, 18786,    12, 13131,
           29,     9,    19,     3, 27692,   642,     8,     3, 30973,   243,
           16,     3,     9,  4263,    18, 24925,  2493,     5,     3, 16911,
         7471,  2744,  4027,    76,  1092,  1954,   133,   991,     8,   412,
            5,   134,     5, 24696,  1446,    13, 18786,     6,    34,   243,
            5, 18786,    47,   788,    12,   942, 26

In [5]:
from torch import optim
from transformers import T5ForConditionalGeneration

model = T5ForConditionalGeneration.from_pretrained(
    pretrained_model_name_or_path="t5-small"
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=1e-5, eps=1e-8)

In [6]:
import numpy as np
from torch import nn


def calc_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)


def train(model, optimizer, dataloader):
    model.train()
    train_loss = 0.0

    for source_ids, source_mask, target_ids, target_mask in dataloader:
        decoder_input_ids = target_ids[:, :-1].contiguous()
        labels = target_ids[:, 1:].clone().detach()
        labels[target_ids[:, 1:] == tokenizer.pad_token_id] = -100

        outputs = model(
            input_ids=source_ids,
            attention_mask=source_mask,
            decoder_input_ids=decoder_input_ids,
            labels=labels,
        )

        loss = outputs.loss
        train_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss = train_loss / len(dataloader)
    return train_loss


def evaluation(model, dataloader):
    with torch.no_grad():
        model.eval()
        val_loss = 0.0

        for source_ids, source_mask, target_ids, target_mask in dataloader:
            decoder_input_ids = target_ids[:, :-1].contiguous()
            labels = target_ids[:, 1:].clone().detach()
            labels[target_ids[:, 1:] == tokenizer.pad_token_id] = -100

            outputs = model(
                input_ids=source_ids,
                attention_mask=source_mask,
                decoder_input_ids=decoder_input_ids,
                labels=labels,
            )

            loss = outputs.loss
            val_loss += loss.item()

    val_loss = val_loss / len(dataloader)
    return val_loss


best_loss = 10000
for epoch in range(epochs):
    train_loss = train(model, optimizer, train_dataloader)
    val_loss = evaluation(model, valid_dataloader)
    print(f"Epoch {epoch + 1}: Train Loss: {train_loss:.4f} Val Loss: {val_loss:.4f}")

    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), "T5ForConditionalGeneration.pt")
        print("Saved the model weights")

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch 1: Train Loss: 4.3126 Val Loss: 3.3205
Saved the model weights
Epoch 2: Train Loss: 3.4287 Val Loss: 2.9186
Saved the model weights
Epoch 3: Train Loss: 3.1448 Val Loss: 2.7672
Saved the model weights
Epoch 4: Train Loss: 2.9918 Val Loss: 2.6777
Saved the model weights
Epoch 5: Train Loss: 2.8885 Val Loss: 2.6148
Saved the model weights
Epoch 6: Train Loss: 2.7996 Val Loss: 2.5662
Saved the model weights
Epoch 7: Train Loss: 2.7480 Val Loss: 2.5268
Saved the model weights
Epoch 8: Train Loss: 2.6929 Val Loss: 2.4953
Saved the model weights
Epoch 9: Train Loss: 2.6557 Val Loss: 2.4686
Saved the model weights
Epoch 10: Train Loss: 2.6048 Val Loss: 2.4470
Saved the model weights
Epoch 11: Train Loss: 2.5575 Val Loss: 2.4257
Saved the model weights
Epoch 12: Train Loss: 2.5307 Val Loss: 2.4091
Saved the model weights
Epoch 13: Train Loss: 2.4887 Val Loss: 2.3950
Saved the model weights
Epoch 14: Train Loss: 2.4690 Val Loss: 2.3840
Saved the model weights
Epoch 15: Train Loss: 2.4332 

In [8]:
model.eval()
with torch.no_grad():
    for source_ids, source_mask, target_ids, target_mask in test_dataloader:
        generated_ids = model.generate(
            input_ids=source_ids,
            attention_mask=source_mask,
            max_length=128,
            num_beams=3,
            repetition_penalty=2.5,
            length_penalty=1.0,
            early_stopping=True,
        )

        for generated, target in zip(generated_ids, target_ids):
            pred = tokenizer.decode(
                generated, skip_special_tokens=True, clean_up_tokenization_spaces=True
            )
            actual = tokenizer.decode(
                target, skip_special_tokens=True, clean_up_tokenization_spaces=True
            )
            print("Generated Headline Text:", pred)
            print("Actual Headline Text   :", actual)
        break

Generated Headline Text: Clinton leads Trump by 4 percentage points in four-war race: poll
Actual Headline Text   : Clinton leads Trump by 4 points in Washington Post: ABC News poll
Generated Headline Text: Democratic senators sharpen potential line of attack against Gorsuch's Supreme Court nomination
Actual Headline Text   : Democrats question independence of Trump Supreme Court nominee
Generated Headline Text: U.S. warns Saudi Arabia that Yemen humanitarian situation could constrain U.S. aid
Actual Headline Text   : In push for Yemen aid, U.S. warned Saudis of threats in Congress
Generated Headline Text: Romanian anti-corruption prosecutors open investigation into Liviu Dragnea on suspicion of forming a criminal group
Actual Headline Text   : Romanian ruling party leader investigated over 'criminal group'
Generated Headline Text: environmental activist endorsed Hillary Clinton for U.S. president
Actual Headline Text   : Billionaire environmental activist Tom Steyer endorses Clinton
G